In [55]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

*STEP 1: LOAD THE RAW*

In [56]:
df = pd.read_excel("school_results.xlsx")

In [57]:
print("Dataset dimensions:", df.shape)
print("\nColumns names:")
print(df.columns.tolist())

Dataset dimensions: (74, 16)

Columns names:
['s/n', 'school', 'student ID', 'class', 'english', 'maths', 'basic science', 'social studies', 'yoruba', 'physical and health education', 'religious knowledge', 'cultural and creative arts', 'agriculture', 'home economics', 'civil', 'computer']


In [58]:
print("\nFirst five records:")
print(df.head())


First five records:
   s/n                            school student ID      class  english  \
0    1  Selected primary school Oyapidan   PUPIL028  Primary 5     78.5   
1    2  Selected primary school Oyapidan   PUPIL029  Primary 5     75.0   
2    3  Selected primary school Oyapidan   PUPIL030  Primary 5     73.0   
3    4  Selected primary school Oyapidan   PUPIL031  Primary 5     74.0   
4    5  Selected primary school Oyapidan   PUPIL032  Primary 5     71.0   

   maths  basic science  social studies  yoruba  \
0   77.0           84.0            84.0    84.0   
1   76.5           86.5            79.5    86.0   
2   73.0           76.5            81.0    80.0   
3   66.0           84.0            81.0    83.0   
4   71.5           83.0            85.0    68.5   

   physical and health education  religious knowledge  \
0                           86.0                 85.0   
1                           80.0                 86.0   
2                           77.5                 8

*Step 2: Clean the dataset*

In [42]:

#Remove spaces around column names

df.columns = df.columns.str.strip()

#Remove completely empty rows
df = df.dropna(how="all")

#Standardise student ID and school name
df["student ID"] = df["student ID"].astype("string").str.strip()

df["school"] = df["school"].astype("string").str.strip()

In [54]:
#Standardis class names
df["class"].astype("string").str.strip().str.upper().replace({
          "P4":"PRIMARY 4",
          "PRI 4":"PRIMARY 4",
          "P5":"PRIMARY 5",
          "PRI 5":"PRIMARY 5",
          "P6":"PRIMARY 6",
          "PRI 6":"PRIMARY 6",
       })

0     PRIMARY 5
1     PRIMARY 5
2     PRIMARY 5
3     PRIMARY 5
4     PRIMARY 5
        ...    
69    PRIMARY 4
70    PRIMARY 4
71    PRIMARY 4
72    PRIMARY 4
73    PRIMARY 4
Name: class, Length: 74, dtype: string

In [44]:
#Replace these with the actual 12 subject column names

subject_cols = [
    "english","maths","basic science","social studies","civil",
    "computer","agriculture","physical and health education","religious knowledge",
    "yoruba","cultural and creative arts","home economics"]

#Convert subject scores to numeric for col in subject_cols:
df[col] = pd.to_numeric(df[col],errors="coerce")

In [45]:
df[subject_cols].dtypes

english                          float64
maths                            float64
basic science                    float64
social studies                   float64
civil                            float64
computer                         float64
agriculture                      float64
physical and health education    float64
religious knowledge              float64
yoruba                           float64
cultural and creative arts       float64
home economics                   float64
dtype: object

In [46]:
print(df.columns.tolist())

['s/n', 'school', 'student ID', 'class', 'english', 'maths', 'basic science', 'social studies', 'yoruba', 'physical and health education', 'religious knowledge', 'cultural and creative arts', 'agriculture', 'home economics', 'civil', 'computer', 'Total Score', 'Average Score', 'Performance Category']


In [47]:
#verify the conversation
for col in subject_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [48]:
#check if any subject has missing or non-numeric scores
df[subject_cols].isnull().sum()

english                          0
maths                            0
basic science                    0
social studies                   0
civil                            0
computer                         0
agriculture                      0
physical and health education    0
religious knowledge              0
yoruba                           0
cultural and creative arts       0
home economics                   0
dtype: int64

In [59]:
#check duplicate student records
print("\nDuplicate student IDs:")
print(df[df.duplicated(subset=["student ID"],
                       keep=False)])


Duplicate student IDs:
    s/n                            school student ID      class  english  \
0     1  Selected primary school Oyapidan   PUPIL028  Primary 5     78.5   
51   52  Selected primary school Oyapidan   PUPIL028  Primary 6     64.0   

    maths  basic science  social studies  yoruba  \
0    77.0           84.0            84.0    84.0   
51   40.0           52.0            51.0    50.0   

    physical and health education  religious knowledge  \
0                            86.0                 85.0   
51                           40.0                 53.0   

    cultural and creative arts  agriculture  home economics  civil  computer  
0                         85.0         82.5            89.5   89.5      84.0  
51                        30.0         50.0            30.0   58.0      65.0  


In [50]:
#check scores outside 0-100 for col in subject_cols:
invalid = df[~df[col].between(0,100)&df[col].notna()]
print(f"{col}:{len(invalid)}invalid scores")

home economics:0invalid scores


*STEP 3: ORGANISE THE DATA*

In [51]:
#calculate total and average

df["Total Score"]=df[subject_cols].sum(axis=1,min_count=1)
df["Average Score"]=df[subject_cols].mean(axis=1).round(2)

In [52]:
#categorise performance
def classify(score):
    if pd.isna(score):
        return "Incomplete Record"
    elif score < 50:
        return "Needs Support"
    elif score < 70:
        return "Satisfactory"
    else:
        return "High Performance"
df["Performance Category"] = df["Average Score"].apply(classify)


*STEP 4: Performance analysis*

In [53]:
print

<function print(*args, sep=' ', end='\n', file=None, flush=False)>